## all current generators and readouts


## setup

In [ ]:
# jupyter setup boilerplate
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from qick import *

# for now, all the tProc v2 classes need to be individually imported (can't use qick.*)

# the main program class
from qick.asm_riscv import RvAveragerProgram as AveragerProgramV2
# for defining sweeps
from qick.asm_v2 import QickSweep1D
# for adding code snippets in loop exec_before/exec_after
from qick.asm_v2 import AsmV2

In [ ]:
# crank up the logging (optional)
import logging
# logging.basicConfig(level=logging.DEBUG)
logging.basicConfig(level=logging.INFO, force=True, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')

## firmware
This demo is using the firmware in https://s3df.slac.stanford.edu/people/meeg/qick/tprocv2/2024-09-03_216_tprocv2r21_demo/.
The firmware project files are also there; the project was compiled with core revision 21 of the tProc v2.
This is a ZCU216 firmware.

This firmware is sort of similar to the old "q3diamond" firmware, in that it has a lot of different generators and readouts (it's intended to cover the full range of currently recommended blocks). Some generators and readouts are new; the old versions are still available for use if you want them in your firmware, and both tProc v1 and v2 support all generators and readouts.
* one standard fullspeed generator (axis_signal_gen_v6)
* two interpolated generators (axis_sg_int4_v2)
    * this is a new version, but the only change from axis_sg_int4_v1 is that the frequency and phase now have 32-bit resolution.
* two different muxed generators, both new
    * axis_sg_mux8_v1 - I'll call this the "fullspeed-mux" generator because it doesn't use the DAC digital mixer
        * it can play up to 8 tones
        * full phase coherence
        * you can set the tone phases
        * it doesn't use the digital mixer, so the DAC can have a sampling frequency up to 10 Gsps; the bad news is that this uses way more FPGA resources and this generator won't fit on the FPGA if you want a lot of other generators as well
    * axis_sg_mixmux8_v1 - I'll call this the "mixer-mux" because it's the same as axis_sg_mux8_v1 but with the digital mixer (similar to the old axis_sg_mux4_v2), so it's 8 tones, fully phase-coherent, configurable phase, goes up to 7 Gsps
        * because it uses 1/4 the FPGA resources of axis_sg_mux8_v1, this will be the preferred mux generator
* one standard readout (axis_readout_v2)
* two different dynamically configured (aka tProc-configured) readouts
    * axis_readout_v3 - I'll call this the "fast dynamic RO," this is the one we've had for a while for phase-reset readout, where you can do a synchronous phase reset on this RO and a generator with 4x the sampling frequency, but it's somewhat quirky and annoying to place in firmware; 
    * axis_dyn_readout_v1 - I'll call this the "dynamic RO," this is a clone of the standard readout but with dynamic control, and might become our new standard readout (the only clear downside is that it consumes a tProc output); much simpler than axis_readout_v3, you can still do synchronous phase reset but the ratio of sampling freqs needs to be 1:2 which is probably less useful
* one muxed readout (axis_pfb_readout_v3) - this is a new one:
    * finer channel spacing, the ADC Nyquist zone is divided into 64 slices instead of 8
    * full phase coherence
    * the ability to take two outputs from a single channel (you will still see lots of crosstalk if you play two tones simultaneously in the same channel; you'll get a warning if your readout frequencies put you at danger of this)
    * you can set the downconversion phase (this is also possible on the standard and tProc-configured readouts)
    * there's a version (axis_pfb_readout_v4) with 8 outputs instead of 4, otherwise identical
 
This firmware also uses the "tProc mux" which allows multiple blocks to be controlled by the same tProc output. Since blocks on a mux share a tProc output and its timed queue, commands to multiple blocks on the same mux must be queued in schedule order, and if you schedule two commands for the same time the second one is going to get delayed.
The tmux will only be needed in firmwares where the total number of generators and dynamic readouts exceeds 16; since there are only 16 DACs, the excess can only come from dynamic ROs. In these cases I plan to tmux the dynamic ROs together (this is safe because RO configuration is usually not timing-critical) with a mux generator (this is safe because mux generators are usually not read out with dynamic RO).

Some general comments (not specific to tProc v2):
* You can always mix and match generators and readouts to suit your RF setup and experimental needs. This demo matches them up in a certain arbitrary way, but this is not the law.
* "Full phase coherence" is my new shorthand for "phase-coherent with respect to frequency changes."
    * All QICK generators and readouts are phase-coherent in the sense that if you play (or downconvert) a pulse, wait a while, and repeat, the carrier LO will maintain its phase just as if it had been running continuously. In terms of qubit control, this means that the carrier phase will track the qubit phase, and your X-gate will always be an X-gate.
    * Some generators and readouts (for example, the standard generator and readout) are fully phase-coherent: if you change the carrier frequency the phase will change in a predictable way, and when you change the frequency back, the phase will go back just as if you had never changed the frequency. This allows you to do frequency scans and get meaningful phase information.
    * Some generators and readouts (for example, the old muxed generator and readout - mux4_v2 and pfb_readout_v2), and the digital mixer, are not fully phase-coherent. These are phase-continuous: if you change the frequency, the phase doesn't jump. But that is not a useful property for us. These are OK for qubit readout once you know what frequencies to use, but you will probably want to do your resonator spectroscopy with a different channel or firmware where you have full phase coherence.
 
***While I have listed the different generators and readouts here, this is not a full description of what each does and how it is used.***
* Please pay attention below where I talk about each and walk through example code.
* I am assuming some prior familiarity with the non-standard generators and readouts here, because my expectation was that people would start by using tProc v2 versions of firmwares they were already familiar with. Things evolved differently. For public release we will need to have "from the ground up" descriptions of each of the blocks, but I did not do that here, sorry.
* *If you want to do in-program sweeps of your readout frequency, you need to use a dynamic readout. That's essentially what "dynamic" means.*

I've set things up with three loopbacks, approximately as shown:
* DAC 0_230 to ADC 0_226, 10MHz-1GHz baluns (fullspeed gen to dynamic RO), with an SMA tee with a short piece of unterminated cable (to act as a resonator) - note that this isn't what's shown in the photo (which is old), I changed baluns, removed the filter, and swapped the barrels for the cable to get a lower frequency.
* DAC 2_230 to ADC 2_226, 10MHz-1GHz baluns (fullspeed-mux gen to mux RO)
* DAC 0_231 to ADC 3_226, 1-4 GHz baluns (interpolated gen to standard RO)
* not cabled: mixer-mux gen on DAC 2_231, fast dynamic RO on ADC 1_226

![image.png](01_gens-and-readouts_photo.jpg)

In [ ]:
from qick.riscq_config import RvConfig
from qick.riscv import _socmap, DEFAULT_SOC
soc = soccfg = RvConfig(_socmap(DEFAULT_SOC))
print(soccfg)

In [ ]:
FSGEN_CH = 1
INTGEN_CH = 1
FSMUXGEN_CH = 3
MIXMUXGEN_CH = 4

DYNRO_CH = 0
MUXRO_CH = [2,3,4,5]
STDRO_CH = 6
FASTDYNRO_CH = 1

## standard (full-speed) gen and dynamically configured readout

If you've used the tProc-configured readout in tProc v1, it's the same here, though I'm calling it "dynamically configured" which is hopefully more descriptive.

A useful feature of dynamic readouts is that you can sweep the downconversion frequency inside your QICK program. In certain firmwares where a dynamic readout shares a fabric clock with a generator, dynamic readouts allow a synchronized phase reset; this is not supported in this demo firmware.

Anyway, the point is that this readout gets its configuration parameters (downconversion frequency, output selection, etc.) through a tProc interface, in the same way that the standard generator gets pulse parameters from the tProc. So this is how you use it:
1. Use `add_readoutconfig()` (similar to `add_pulse()`) to define a "readout config" in your program that is a lot like a pulse (it's stored as a waveform, it can be swept).
2. At some point in your program, use `send_readoutconfig()` (similar to `pulse()`) to send the config to the readout. Make sure this reaches your readout before the signal does.

Don't forget to do both steps! Your readout is not going to magically see the readout config just because you defined it. Nor will it get sweep updates - if you sweep a readout parameter, you need to send the config to your readout at each sweep step (typically you can just do this at start-of-shot).

By the way - for the other ("PYNQ-configured") readouts, you put your readout parameters in declare_readout(). Those parameters do not go in the assembly code; they are written directly to the readout block (through the PYNQ interface) before the program is executed, and you can't change those parameters while your program is running. You can make an analogy to the mux generator's frequency+gain parameters (which are set through PYNQ).

### a little note about how/why the fast dynamic readout is weird
The dynamic readout comes in two flavors, the "standard" dynamic readout (axis_dyn_readout_v1) and the "fast" dynamic readout (axis_readout_v3). The dynamic RO was designed as a phase-resettable readout for applications (e.g. the three-wave mixing readout in https://arxiv.org/abs/2311.17171, or the laser stabilization in https://github.com/openquantumhardware/qick/issues/120) where you need to phase-reset the generator that's playing your readout tone, and therefore need to simultaneously reset the readout phase. This only works if the gen and RO share their fabric clock. With the standard dynamic RO, this implies fs\_DAC = 2\*fs\_ADC; for ZCU216 this would mean a generator sampling frequency of <5 Gsps, which is not great for many applications. The fast dynamic RO doubles the fabric clock, and therefore expects fs\_DAC = 4\*fs\_ADC, which lets you run both the gen+RO at the ZCU216's maximum speed.

The clock doubling makes the fast dynamic readout a little weird:
* Compared to the other readouts, it's relatively annoying to put this one in a firmware.
* It only downsamples the ADC by 4 as opposed to 8. This improves your time resolution but raises the cutoff of the decimation filter.
    * You probably don't care about the time resolution.
    * You may care about the decimation cutoff, because you rely on that filter to kill the image tone described in https://github.com/openquantumhardware/qick/blob/main/qick_demos/08_Special_buffers.ipynb. In other words, you need more separation between your readout tone and the ADC's Nyquist zone boundaries: roughly fs/16 instead of fs/32 (https://github.com/openquantumhardware/qick/issues/120#issuecomment-1494931618).
 
<!-- If there is big demand for a sweepable readout that isn't weird but doesn't have a synchronized phase-reset, it's doable - let us know. (Technically you could still sync the phase-reset, but with a requirement of  as opposed to fs\_DAC = 4\*fs\_ADC, and most people want their DACs to be faster than that.) -->

### basic loopback
Here the readout parameters aren't swept, so we can both define them and set them in `_initialize()`.

Note that I put definitions (these add stuff to the program metadata - generally they start with "add" or "declare") before the tProc instructions (these add ASM instructions to the program, which are executed by the tProc) in `_initialize()`. This isn't a rule, but it's good code hygiene to sort of separate the "setup" stuff from the code that actually runs as part of your experiment sequence. (As an example of what not to do? You can put definitions in the `_body()` and your program will work fine! Definitions do their thing when the Python interpreter sees them during compilation, so your tProc loops have no effect on them. But it will look very strange.)

In [ ]:
class SimpleSweepProgram2(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=cfg['nqz'])
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        self.add_loop("myloop", self.cfg["steps"])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['freq'], gen_ch=gen_ch)

        self.add_gauss(ch=gen_ch, name="ramp", sigma=cfg['ramp_len']/10, length=cfg['ramp_len'], even_length=True)
        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch, 
                       style="flat_top", 
                       envelope="ramp", 
                       freq=cfg['freq'], 
                       length=cfg['flat_len'],
                       phase=cfg['phase'],
                       gain=cfg['gain'], 
                      )

        
        self.send_readoutconfig(ch=cfg['ro_ch'], name="myro", t=0)
        
    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
        self.trigger(ros=[cfg['ro_ch']], t=cfg['trig_time'])

In [ ]:
# do a sweep with 5 points and plot decimated
config = {'steps': 5,
          'gen_ch': FSGEN_CH,
          'ro_ch': DYNRO_CH,
          'freq': 1500,
          'nqz': 1,
          'trig_time': 0.4,
          'ro_len': 0.3,
          'flat_len': 0.05,
          'ramp_len': 0.2,
          'phase': QickSweep1D("myloop", -360, 720),
          'gain': QickSweep1D("myloop", 0.0, 1.0)
         }

prog = SimpleSweepProgram2(soccfg, reps=1, final_delay=0.5, cfg=config)

iq_list = prog.acquire_decimated(soc, rounds=1)
t = prog.get_time_axis(ro_index=0)

for ii, iq in enumerate(iq_list[0]):
    plt.plot(t, iq[:,0], label="I value, step %d"%(ii))
    # plt.plot(t, iq[:,1], label="Q value, step %d"%(ii))
    # plt.plot(t, np.abs(iq.dot([1,1j])), label="mag, step %d"%(ii))
plt.legend()
plt.ylabel("amplitude [ADU]")
plt.xlabel("time [us]");


### frequency sweeping
Now we'll sweep the upconversion and downconversion frequencies together. Note that we therefore need to run `send_readoutconfig()` inside the loop where the frequency gets swept; we accomplish this by putting it in that loop's `exec_before`, though you could also put it at the beginning of your `_body()`.

We'll start with a small frequency range.

In [ ]:
START_FREQ = 100
CAL_RANGE = 1
END_FREQ = 8000

class FreqSweepProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=cfg['nqz'])
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['freq'], gen_ch=gen_ch)
        
        loopbefore = AsmV2()
        # if you delay the config by too long, you can see the readout get reconfigured in the middle of your pulse
        loopbefore.send_readoutconfig(ch=cfg['ro_ch'], name="myro", t=0)
        self.add_loop("myloop", self.cfg["steps"], exec_before=loopbefore)

        self.add_gauss(ch=gen_ch, name="ramp", sigma=cfg['ramp_len']/10, length=cfg['ramp_len'], even_length=True)
        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch, 
                       style="flat_top", 
                       envelope="ramp", 
                       freq=cfg['freq'], 
                       length=cfg['flat_len'],
                       phase=cfg['phase'],
                       gain=cfg['gain'], 
                      )
        
    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
        self.trigger(ros=[cfg['ro_ch']], t=cfg['trig_time'])

# do a sweep with 5 points and plot decimated
config = {'steps': 5,
          'gen_ch': FSGEN_CH,
          'ro_ch': DYNRO_CH,
          'freq': QickSweep1D("myloop", START_FREQ, START_FREQ+CAL_RANGE),
          'nqz': 1,
          'trig_time': 0.4,
          'ro_len': 0.3,
          'flat_len': 0.05,
          'ramp_len': 0.2,
          'phase': 0,
          'gain': 1.0
         }

prog = FreqSweepProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

iq_list = prog.acquire_decimated(soc, rounds=10)
t = prog.get_time_axis(ro_index=0)

for ii, iq in enumerate(iq_list[0]):
    plt.plot(t, iq[:,0], label="I value, step %d"%(ii))
    # plt.plot(t, iq[:,1], label="Q value, step %d"%(ii))
    # plt.plot(t, np.abs(iq.dot([1,1j])), label="mag, step %d"%(ii))
plt.legend()
plt.ylabel("amplitude [ADU]")
plt.xlabel("time [us]");

In [ ]:
print(config['freq'].get_actual_values({k:n for k,n,_,_ in prog.loops}))
print(prog.get_pulse_param("mypulse","freq", as_array=True))


Now let's do small steps. As you know from the "phase coherence" demo (demo 01 in qick_demos), frequency sweeps give you a rotating phase because the upconversion and downconversion LOs have different start times.

In [ ]:
config['steps']=2797

prog = FreqSweepProgram(soccfg, reps=10, final_delay=1.0, cfg=config)
freqs = prog.get_pulse_param('myro', 'freq', as_array=True)
iq_list = prog.acquire(soc, rounds=1, progress=True)
# plt.plot(np.angle(iq_list[0][0].dot([1,1j]), deg=True))
plt.plot(iq_list[0][0,:,0], iq_list[0][0,:,1], '.')
plt.ylabel("Q [ADU]")
plt.xlabel("I [ADU]");

We can plot the phase vs. frequency. Note that because our sweep steps are very small, and the sweep step needs to be rounded to the frequency-matching step size, the sweep range is noticeably different from what we asked for. (If this was a problem, you could adjust the step count to bring your nominal step size closer to the rounded value.)

Anyway, the phase difference between consecutive points is very small, which gives us confidence that we're measuring the phase rotation correctly and not getting something aliased.

In [ ]:
iq_complex = iq_list[0][0].dot([1,1j])
phases = np.unwrap(np.angle(iq_complex))/(2*np.pi)

plt.plot(freqs, phases)
plt.ylabel("Phase [cycles]")
plt.xlabel("Frequency [MHz]")

print("phase step (cycles):", phases[1]-phases[0])

We can do a linear least-squares fit to get the slope, which we can interpret as a delay. (Your VNA would call this an "electrical delay" calibration.)

In [ ]:
a = np.vstack([freqs, np.ones_like(freqs)]).T
phase_delay = np.linalg.lstsq(a, phases, rcond=None)[0][0]
print("relative delay %.4f us"%(phase_delay))

Now let's do a wide-range sweep and use the calibration to remove the phase delay.

In [ ]:
config['freq'] = QickSweep1D("myloop", START_FREQ, END_FREQ)
config['steps'] = 10001
# freqs = np.linspace(START_FREQ, END_FREQ, config['steps'])

prog = FreqSweepProgram(soccfg, reps=100, final_delay=1.0, cfg=config)
freqs = prog.get_pulse_param('myro', 'freq', as_array=True)
iq_list = prog.acquire(soc, rounds=1, progress=True)
iq_complex = iq_list[0][0].dot([1,1j])


iq_rotated = iq_complex*np.exp(-1j*freqs*2*np.pi*phase_delay)
plt.plot(np.real(iq_rotated), np.imag(iq_rotated))
plt.ylabel("Q [ADU]")
plt.xlabel("I [ADU]");

Plot the amplitude and phase: we can see the resonator at 940 MHz as a dip in magnitude and a pi rotation in the phase. As an open-ended quater-wave resonator, it resonates at all odd multiples of 940 MHz and you also see it at 2.8 GHz, 4.7 GHz, etc.

By the way - notice that the spectrum is mirrored around 4.9 GHz. This happens because the DAC sampling frequency is 4 times that of the ADC, and 4.9 GHz is a Nyquist zone boundary for both. (The old ZCU111 standard firmware also had this property; the DAC sampling frequency was double that of the ADC.) In general, DACs and ADCs cannot distinguish between Nyquist images (that's essentially the definition of a Nyquist zone). So:
* Whether you ask the generator to play 4.9-x or 4.9+x, the DAC will play both. Whether you ask the readout to downconvert at 4.9-x or 4.9+x, it will be sensitive to both. The first tone is in the DAC's 1st Nyquist zone and the ADC's 4th/3rd/2nd/1st Nyquist zone, the second is in the DAC's 2nd Nyquist zone and the ADC's 5th/6th/7th/8th.
* The transmission value you measure will be the sum of the two tones.
* Filtering (the DAC's `nqz` setting, the analog response of the DAC and ADC, the balun response, and any other analog filtering) changes the relative strength and transmission of the two tones. Here the resonance is in a region where the `nqz` setting, analog response, and balun response strongly favor the 940 MHz tone over the 9 GHz tone, so we mostly just see the 940 MHz response. Were that not the case, the resonance would be largely washed out by transmission at the other tone.

More generally, in a serious measurement you always want to filter, even if your firmware does not have this simple-multiples relation between DAC and ADC sampling frequencies:
* Filter your DAC to remove any unwanted Nyquist images which could otherwise saturate your amplifiers, create spurs in your mixers, and do weird things to your device.
* Filter your ADC to ensure that it sees your measurement tone but is blind to the noise at any of its Nyquist images.
* If your operating frequency is close to a DAC or ADC Nyquist boundary, it's going to be very difficult to filter appropriately. You should consider changing the sampling frequency (this requires recompiling the firmware).

In [ ]:
phases_corrected = np.unwrap(np.angle(iq_rotated))/(2*np.pi)

fig, plot = plt.subplots()
plot.semilogy(freqs, np.abs(iq_rotated), label="amplitude")
ax2 = plot.twinx()  # instantiate a second axes that shares the same x-axis
ax2.plot(freqs, phases_corrected, color='red', label='phase')

plot.set_ylabel("amplitude [ADU]")
plot.set_xlabel("frequency [MHz]")
ax2.set_ylabel("phase [cycles]");

Note that instead of correcting the phase of the data, you could use a phase sweep in the pulse or the readout config. There's no obvious advantage in this case (or ever?), so it's left as an exercise for the reader.